# LLM Inference Learning Lab — Results & Analysis

**Owner:** Mohamed — Senior Engineering Manager, AWS Auto Scaling  
**Goal:** Build hands-on understanding of LLM inference infrastructure for Anthropic interview prep  
**Repo:** [github.com/mrefaat87/vLLM-Inference-Lab](https://github.com/mrefaat87/vLLM-Inference-Lab)

---

## Test Index

| # | Test Name | Stage | What It Proves |
|---|-----------|-------|---------------|
| 1 | [Ollama Sequential Queuing](#test-1) | S1 | Ollama processes requests one at a time |
| 2 | [vLLM Continuous Batching](#test-2) | S2 | vLLM processes requests in parallel |
| 3 | [Quantization Speed Comparison](#test-3) | S2 | AWQ 4-bit vs AWQ_Marlin vs GPTQ Int8 |
| 4 | [Quantization Quality Comparison](#test-4) | S2 | 4-bit and 8-bit produce identical quality |
| 5 | [Max Concurrency Scaling](#test-5) | S2 | System throughput peaks at N=60, then declines |
| 6 | [Variable Prompt Scheduling](#test-6) | S2 | vLLM prioritizes short-prefill requests naturally |
| 7 | [KV Cache Cliff (Long Context)](#test-7) | S2 | Prefix caching masks KV pressure |
| 8 | [Preemption & Queuing](#test-8) | S2 | KV cache at 100% → vLLM queues, then crashes |
| 9 | [Prefill vs Decode Bottlenecks](#test-9) | S2 | Prefill scales with input; decode scales with output |
| 10 | [Why Output Tokens Cost More](#test-10) | S2 | Output tokens cost 16.9x more GPU time |

---

**How to use this notebook:**
- Each test has: scenario, chart(s), insights, and a "Your Notes" cell you can edit
- All charts read from JSON files — re-run cells after new experiments to update
- Reference tests by number in chat (e.g., "look at Test 5, why does throughput drop at N=70?")

In [ ]:
# Setup — run this cell first
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

# Configure matplotlib for clean charts
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

BASE = os.path.dirname(os.path.abspath('inference_lab_results.ipynb'))

def load_json(filename):
    with open(os.path.join(BASE, filename)) as f:
        return json.load(f)

# Load all results
results = load_json('stage2_results.json')
max_conc = load_json('stage2_max_concurrency_results.json')
var_load = load_json('stage2_variable_load_results.json')
exp2 = load_json('stage2_exp2_results.json')
exp3 = load_json('stage2_exp3_results.json')
cliff = load_json('stage2_cliff_results.json')
preemption = load_json('stage2_exp1_preemption_results.json')

print("All data loaded successfully.")

<a id="test-1"></a>
## Test 1: Ollama Sequential Queuing (Stage 1)

### Scenario
3 concurrent requests sent to Ollama on Apple M4 (16GB), serving Qwen2.5:7B (Q4). Ollama has no batching — it processes one request at a time. We measured whether requests run in parallel or queue sequentially.

### Key Concepts
- **Ollama = single-worker server.** Like a single-instance target group with no ASG.
- **Gap between first-tokens ≈ single request duration** proves sequential processing.
- Hardware: Apple M4 with llama.cpp backend → 0.3 tok/s (memory bandwidth bottleneck + non-native Metal runtime).

In [ ]:
# Test 1: Ollama timeline waterfall
ollama = results['load_tests']['ollama_local_m4']
reqs = sorted(ollama['requests'], key=lambda r: r['first_token'])

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['#e74c3c', '#3498db', '#2ecc71']

for i, r in enumerate(reqs):
    # Waiting phase (start → first token)
    ax.barh(i, r['first_token'] - r['start'], left=r['start'], 
            color=colors[i], alpha=0.3, height=0.5, label='Waiting/Prefill' if i == 0 else '')
    # Generating phase (first token → end)
    ax.barh(i, r['end'] - r['first_token'], left=r['first_token'],
            color=colors[i], alpha=0.8, height=0.5, label='Generating' if i == 0 else '')
    # TTFT marker
    ax.plot(r['first_token'], i, 'k|', markersize=15, markeredgewidth=2)
    ax.text(r['end'] + 2, i, f"{r['total']:.0f}s total, {r['tokens']} tok", va='center', fontsize=9)

ax.set_yticks(range(len(reqs)))
ax.set_yticklabels([f"Req {r['id']}" for r in reqs])
ax.set_xlabel('Time (seconds from test start)')
ax.set_title('Test 1: Ollama Sequential Queuing — Each Request Waits for the Previous to Finish')

# Add annotations for the gaps
for i in range(1, len(reqs)):
    gap = reqs[i]['first_token'] - reqs[i-1]['first_token']
    mid = (reqs[i]['first_token'] + reqs[i-1]['first_token']) / 2
    ax.annotate(f'gap: {gap:.0f}s', xy=(mid, i-0.5), fontsize=9, ha='center',
                color='red', fontweight='bold')

ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f"TTFT spread: {ollama['summary']['ttft_spread']:.1f}s")
print(f"Avg gap between first-tokens: {ollama['summary']['avg_gap_between_first_tokens']:.1f}s")
print(f"Gap/Total ratio: {ollama['summary']['gap_total_ratio']} (1.0 = pure sequential)")

### Insights
- Staircase pattern: each request starts only after the previous **fully completes** (prefill + all decode)
- Gap between first-tokens (~81s) matches single request duration — textbook FIFO queuing
- Request 2's TTFT of 177s is pure **queuing delay**, not model slowness
- 0.3 tok/s is due to llama.cpp's inefficient Metal backend on M4 (MLX would give ~20 tok/s)

**Auto Scaling analogy:** Single-instance target group. The "latency" problem isn't processing speed — it's queue depth. Exactly the problem horizontal scaling (or batching) solves.

### Your Notes
*Add your own observations here*

<a id="test-2"></a>
## Test 2: vLLM Continuous Batching (Stage 2)

### Scenario
5 concurrent requests sent to vLLM on T4 GPU, serving Qwen2.5-7B-AWQ (4-bit). vLLM uses continuous batching — requests are processed in parallel. Same prompt as Test 1 for direct comparison.

### Key Concepts
- **Continuous batching = city bus.** Picks up passengers at every stop, doesn't wait for everyone to board.
- **Gap/Total ratio near 0** proves true parallel processing.
- All requests get their first token within milliseconds of each other.

In [ ]:
# Test 2: vLLM vs Ollama side-by-side waterfall
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4), gridspec_kw={'width_ratios': [3, 1]})

# Left: vLLM waterfall
vllm_data = results['load_tests']['vllm_awq_4bit']
vllm_reqs = sorted(vllm_data['requests'], key=lambda r: r['first_token'])
colors5 = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

for i, r in enumerate(vllm_reqs):
    ax1.barh(i, r['first_token'] - r['start'], left=r['start'],
            color=colors5[i], alpha=0.3, height=0.5)
    ax1.barh(i, r['end'] - r['first_token'], left=r['first_token'],
            color=colors5[i], alpha=0.8, height=0.5)
    ax1.plot(r['first_token'], i, 'k|', markersize=15, markeredgewidth=2)

ax1.set_yticks(range(len(vllm_reqs)))
ax1.set_yticklabels([f"Req {r['id']}" for r in vllm_reqs])
ax1.set_xlabel('Time (seconds)')
ax1.set_title('vLLM: All 5 Requests Process in Parallel')

# Right: comparison bar chart
metrics = ['TTFT spread\n(seconds)', 'Gap/Total\nratio', 'Tok/s\n(per req)']
ollama_vals = [162.9, 1.0, 0.3]
vllm_vals = [0.176, 0.01, 44.0]

x = np.arange(len(metrics))
width = 0.35
bars1 = ax2.bar(x - width/2, ollama_vals, width, label='Ollama', color='#e74c3c', alpha=0.7)
bars2 = ax2.bar(x + width/2, vllm_vals, width, label='vLLM', color='#3498db', alpha=0.7)
ax2.set_xticks(x)
ax2.set_xticklabels(metrics)
ax2.set_yscale('log')
ax2.set_title('Ollama vs vLLM')
ax2.legend()

# Add value labels
for bar, val in zip(bars1, ollama_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.3, 
             f'{val}', ha='center', va='bottom', fontsize=8, fontweight='bold')
for bar, val in zip(bars2, vllm_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.3,
             f'{val}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

### Insights
- TTFT spread: 162.9s (Ollama) → 0.176s (vLLM) — **925x improvement**
- Tok/s: 0.3 (Ollama on M4) → 44 (vLLM on T4) — hardware + batching combined
- Gap/Total ratio: 1.0 → 0.01 — from pure sequential to true parallel
- Prefix caching gave requests 2-4 lower TTFT (~0.55s vs ~0.71s) — they reused KV cache from request 0

**Auto Scaling analogy:** Adding continuous batching is like going from a single-worker Flask app to a multi-threaded server. The hardware didn't change — the scheduling did.

### Your Notes
*Add your own observations here*

---

<a id="test-3"></a>
## Test 3: Quantization Speed Comparison (Stage 2)

### Scenario
Same 5-concurrent-request load test, same T4 GPU, three different quantization configs:
- **AWQ 4-bit:** ~4GB model, generic INT4 kernel
- **AWQ_Marlin 4-bit:** same weights, optimized Marlin kernel for faster matrix multiply
- **GPTQ Int8:** ~8.3GB model, 8-bit weights

### Key Concepts
- **AWQ vs AWQ_Marlin** = same data format, different compute kernel (like JPEG file vs GPU-accelerated decoder)
- **4-bit vs 8-bit** = smaller model → more KV room → more concurrency, but potentially lower quality
- Decode is memory-bandwidth bound → 2x larger model → ~2x slower per token

In [ ]:
# Test 3: Quantization speed comparison
configs = {
    'AWQ\n4-bit': results['load_tests']['vllm_awq_4bit']['summary'],
    'AWQ Marlin\n4-bit': results['load_tests']['vllm_awq_marlin_4bit']['summary'],
    'GPTQ\nInt8': results['load_tests']['vllm_gptq_int8']['summary'],
}

# Additional data not in summary
extra = {
    'AWQ\n4-bit': {'tok_s': 44.0, 'ttft_min': 0.547, 'avg_tbt': 0.0225, 'vram': 4.0, 'kv_room': 10.0},
    'AWQ Marlin\n4-bit': {'tok_s': 45.3, 'ttft_min': 0.291, 'avg_tbt': 0.0221, 'vram': 4.0, 'kv_room': 9.3},
    'GPTQ\nInt8': {'tok_s': 27.0, 'ttft_min': 0.831, 'avg_tbt': 0.0375, 'vram': 8.3, 'kv_room': 4.2},
}

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
names = list(configs.keys())
colors3 = ['#3498db', '#2ecc71', '#e74c3c']

# Chart 1: Tok/s per request
vals = [extra[n]['tok_s'] for n in names]
axes[0].bar(names, vals, color=colors3, alpha=0.8)
axes[0].set_title('Decode Speed\n(tok/s per request)')
for i, v in enumerate(vals):
    axes[0].text(i, v + 1, f'{v}', ha='center', fontweight='bold')

# Chart 2: TTFT
vals = [extra[n]['ttft_min'] for n in names]
axes[1].bar(names, vals, color=colors3, alpha=0.8)
axes[1].set_title('TTFT (best case)\n(seconds)')
for i, v in enumerate(vals):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Chart 3: Model VRAM vs KV Cache room
vram = [extra[n]['vram'] for n in names]
kv = [extra[n]['kv_room'] for n in names]
x = np.arange(len(names))
axes[2].bar(x, vram, 0.4, label='Model Weights', color='#e74c3c', alpha=0.7)
axes[2].bar(x, kv, 0.4, bottom=vram, label='KV Cache Room', color='#2ecc71', alpha=0.7)
axes[2].set_xticks(x)
axes[2].set_xticklabels(names)
axes[2].set_ylabel('GB')
axes[2].set_title('VRAM Allocation\n(16GB T4)')
axes[2].axhline(y=16, color='black', linestyle='--', alpha=0.5, label='T4 Total')
axes[2].legend(fontsize=8)

# Chart 4: Avg TBT (time between tokens)
vals = [extra[n]['avg_tbt'] * 1000 for n in names]  # convert to ms
axes[3].bar(names, vals, color=colors3, alpha=0.8)
axes[3].set_title('Time Between Tokens\n(ms)')
for i, v in enumerate(vals):
    axes[3].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

plt.suptitle('Test 3: Quantization Speed Comparison — Same Model, Same Hardware', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Insights
- Marlin kernel: ~3% faster decode, **47% faster prefill** (TTFT 0.29s vs 0.55s). Same weights, better kernel.
- Int8 is ~40% slower on decode — model is 2x larger, memory bandwidth is the bottleneck
- Int8 uses 2x VRAM → half the KV cache room → half the max concurrency
- All three batch perfectly (gap/total ≈ 0) — batching behavior is engine-level, not quantization-level

**Auto Scaling analogy:** Like choosing instance types — c5.xlarge (4-bit, fast, more headroom) vs m5.2xlarge (8-bit, slower, less room per dollar).

### Your Notes
*Add your own observations here*

---

<a id="test-4"></a>
## Test 4: Quantization Quality Comparison (Stage 2)

### Scenario
60-question benchmark suite across 7 categories (Math, Factual Recall, Instruction Following, Code Gen, MMLU-style, GSM8K-style, HumanEval-style). Same questions run against both AWQ INT4 and GPTQ INT8. Temperature=0 for reproducibility.

In [ ]:
# Test 4: Quality comparison — accuracy + latency by category
qb = results['quality_benchmarks']
categories = list(qb['awq_int4']['categories'].keys())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Chart 1: Accuracy (should be identical — both 100%)
x = np.arange(len(categories))
width = 0.35
int4_acc = [qb['awq_int4']['categories'][c]['accuracy'] for c in categories]
int8_acc = [qb['gptq_int8']['categories'][c]['accuracy'] for c in categories]

ax1.bar(x - width/2, int4_acc, width, label='AWQ INT4', color='#3498db', alpha=0.8)
ax1.bar(x + width/2, int8_acc, width, label='GPTQ INT8', color='#e74c3c', alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels([c.replace(' ', '\n') for c in categories], fontsize=8)
ax1.set_ylabel('Accuracy (%)')
ax1.set_ylim(0, 110)
ax1.set_title('Quality: Both Score 100% — No Difference')
ax1.legend()

# Chart 2: Latency comparison — INT4 is consistently faster
int4_lat = [qb['awq_int4']['categories'][c]['avg_latency_s'] for c in categories]
int8_lat = [qb['gptq_int8']['categories'][c]['avg_latency_s'] for c in categories]

ax2.bar(x - width/2, int4_lat, width, label='AWQ INT4', color='#3498db', alpha=0.8)
ax2.bar(x + width/2, int8_lat, width, label='GPTQ INT8', color='#e74c3c', alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels([c.replace(' ', '\n') for c in categories], fontsize=8)
ax2.set_ylabel('Avg Latency (seconds)')
ax2.set_title('Speed: INT4 is ~60% Faster Across All Categories')
ax2.legend()

# Add percentage labels
for i in range(len(categories)):
    if int4_lat[i] > 0:
        pct = ((int8_lat[i] / int4_lat[i]) - 1) * 100
        ax2.text(i, max(int4_lat[i], int8_lat[i]) + 0.2, f'+{pct:.0f}%', 
                ha='center', fontsize=8, color='red')

plt.suptitle('Test 4: Same Quality, Different Speed — The Case for INT4', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"INT4 overall: {qb['awq_int4']['overall']['pass']}/{qb['awq_int4']['overall']['total']} "
      f"({qb['awq_int4']['overall']['accuracy']}%) — avg {qb['awq_int4']['overall']['avg_latency_s']:.2f}s")
print(f"INT8 overall: {qb['gptq_int8']['overall']['pass']}/{qb['gptq_int8']['overall']['total']} "
      f"({qb['gptq_int8']['overall']['accuracy']}%) — avg {qb['gptq_int8']['overall']['avg_latency_s']:.2f}s")

### Insights
- Both INT4 and INT8 scored **60/60 (100%)** — zero quality difference on this benchmark
- INT4 is ~60% faster across every category — the speed advantage is consistent
- Code gen and GSM8K (long outputs) show the biggest absolute time difference
- Q54 (train meeting problem) flagged as likely false positive — substring scorer may have matched incorrectly
- Quality degradation from 4-bit shows up on smaller models (1-3B), extreme quantization (2-bit), or adversarial evals — not at 7B scale

**Production takeaway:** At 7B+ scale, INT4 with modern quantization (AWQ/GPTQ) is the clear winner — same quality, 60% faster, 2x more KV cache room for concurrency.

### Your Notes
*Add your own observations here*

---

<a id="test-5"></a>
## Test 5: Max Concurrency Scaling (Stage 2)

### Scenario
Ramp identical concurrent requests from 1 → 80 on AWQ_Marlin 4-bit. Same short prompt (~30 token output). vLLM reports theoretical max of 62 concurrent at 2048 context, but our short responses use only ~30 tokens of KV each.

We want to see: how does system throughput scale? Where does per-request latency degrade? Is there a cliff?

In [ ]:
# Test 5: Max concurrency — dual axis chart
valid = [r for r in max_conc if 'error_sample' not in r]
ns = [r['concurrency'] for r in valid]
sys_tok = [r['system_tok_s'] for r in valid]
req_tok = [r['per_request_tok_s_avg'] for r in valid]
ttft_p50 = [r['ttft_p50'] for r in valid]
ttft_max = [r['ttft_max'] for r in valid]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Throughput — system vs per-request (dual axis)
color1, color2 = '#3498db', '#e74c3c'
ax1.plot(ns, sys_tok, 'o-', color=color1, linewidth=2, markersize=8, label='System tok/s')
ax1.set_xlabel('Concurrent Requests')
ax1.set_ylabel('System Throughput (tok/s)', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax1b = ax1.twinx()
ax1b.plot(ns, req_tok, 's--', color=color2, linewidth=2, markersize=8, label='Per-request tok/s')
ax1b.set_ylabel('Per-Request Speed (tok/s)', color=color2)
ax1b.tick_params(axis='y', labelcolor=color2)

# Mark peak
peak_idx = sys_tok.index(max(sys_tok))
ax1.annotate(f'Peak: {max(sys_tok):.0f} tok/s\nat N={ns[peak_idx]}', 
            xy=(ns[peak_idx], max(sys_tok)), xytext=(ns[peak_idx]-15, max(sys_tok)+50),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10, fontweight='bold')

# Mark 50% degradation point
base = req_tok[0]
for i, v in enumerate(req_tok):
    if v < base * 0.5:
        ax1b.axvline(x=ns[i], color='gray', linestyle=':', alpha=0.5)
        ax1b.text(ns[i]+1, v+2, f'50% degradation\n(N={ns[i]})', fontsize=8, color='gray')
        break

ax1.set_title('Throughput: System Scales Up, Per-Request Degrades')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

# Chart 2: TTFT progression
ax2.fill_between(ns, ttft_p50, ttft_max, alpha=0.3, color='#f39c12', label='p50–max range')
ax2.plot(ns, ttft_p50, 'o-', color='#f39c12', linewidth=2, label='TTFT p50')
ax2.plot(ns, ttft_max, 's--', color='#e67e22', linewidth=1, alpha=0.7, label='TTFT max')

# Highlight the N=10 CUDA compilation spike
spike_idx = next(i for i, n in enumerate(ns) if n == 10)
if ttft_p50[spike_idx] > 5:
    ax2.annotate('CUDA graph\ncompilation\n(one-time)', 
                xy=(10, ttft_p50[spike_idx]), xytext=(20, ttft_p50[spike_idx]-1),
                arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')

ax2.set_xlabel('Concurrent Requests')
ax2.set_ylabel('TTFT (seconds)')
ax2.set_title('TTFT: Stable Except for One-Time Compilation Spike')
ax2.legend()

plt.suptitle('Test 5: How Concurrency Affects Throughput and Latency', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Insights
- **System throughput peaks at N=60 (1132 tok/s)** then declines — scheduler overhead outweighs batching benefit
- **Per-request tok/s halves by N=40** (49.5 → 24.0) — GPU bandwidth is shared across the batch
- **N=10 TTFT spike (9.1s)** is a one-time CUDA graph compilation cost — like Lambda cold start
- **Zero failures even at N=80** — KV usage peaked at 1.7% because responses were only ~30 tokens
- **PagedAttention** means theoretical max (62 at 2048 ctx) vastly understates capacity for short responses

**Auto Scaling analogy:** System throughput curve = the reason you don't set ASG max to infinity. There's an optimal fleet size where coordination overhead starts exceeding capacity gains.

### Your Notes
*Add your own observations here*

---

<a id="test-6"></a>
## Test 6: Variable Prompt Scheduling (Stage 2)

### Scenario
Mixed workload: Short (~15 input tokens, 20 output), Medium (~80 input, 150 output), Long (~300 input, 300 output) requests sent concurrently. Ramp from 5 → 60. We want to see: does vLLM prioritize any prompt type, and how does TTFT degrade differently by type?

In [ ]:
# Test 6: Variable prompt scheduling — TTFT by type + scheduling order
# Filter to mixed rounds only (skip pure S/M/L rounds)
mixed = [r for r in var_load if len(r['prompt_mix']) > 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: TTFT by prompt type across concurrency levels
for ptype, color, marker in [('short', '#2ecc71', 'o'), ('medium', '#f39c12', 's'), ('long', '#e74c3c', '^')]:
    ns_type = []
    ttft_type = []
    for r in mixed:
        if ptype in r['by_type']:
            ns_type.append(r['concurrency'])
            ttft_type.append(r['by_type'][ptype]['ttft_p50'])
    ax1.plot(ns_type, ttft_type, f'{marker}-', color=color, linewidth=2, markersize=8, 
            label=f'{ptype.capitalize()} prompt')

ax1.set_xlabel('Total Concurrent Requests')
ax1.set_ylabel('TTFT p50 (seconds)')
ax1.set_title('TTFT by Prompt Type — Short Prompts Barely Degrade')
ax1.legend()

# Chart 2: Scheduling order visualization
# Show the first-token delivery order as a color-coded bar
order_data = [(r['concurrency'], r['scheduling_order_first20']) for r in mixed if r['concurrency'] >= 10]

color_map = {'S': '#2ecc71', 'M': '#f39c12', 'L': '#e74c3c'}
for idx, (n, order) in enumerate(order_data):
    for j, char in enumerate(order):
        ax2.barh(idx, 1, left=j, color=color_map.get(char, 'gray'), 
                edgecolor='white', linewidth=0.5, height=0.7)

ax2.set_yticks(range(len(order_data)))
ax2.set_yticklabels([f'N={n}' for n, _ in order_data])
ax2.set_xlabel('Position in first-token delivery order')
ax2.set_title('Scheduling Order — Who Gets First Token First?')

# Legend
patches = [mpatches.Patch(color='#2ecc71', label='Short'),
           mpatches.Patch(color='#f39c12', label='Medium'),
           mpatches.Patch(color='#e74c3c', label='Long')]
ax2.legend(handles=patches, loc='lower right')

plt.suptitle('Test 6: vLLM Scheduling Under Mixed Workload', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Insights
- **Short prompts barely degrade** (0.23s → 0.28s, +20%) while **Medium/Long degrade ~85%** (0.23s → 0.43s)
- Scheduling order reveals **emergent priority**: shorts always get first token first — not by explicit priority, but because chunked prefill finishes faster for shorter prompts
- At N=50-60, the first 19 tokens delivered are ALL short prompts before any medium gets through
- Prefix cache hit rate climbed to 73% — shared prompts get reused across rounds
- Zero queuing, zero preemption even at N=60 — KV peaked at 3.9%

**Auto Scaling analogy:** Like two instance types launching simultaneously — t3.micro boots in 15s while c5.4xlarge takes 90s. Shorter startup = faster time-to-serve. Not priority — just less work to do.

### Your Notes
*Add your own observations here*

---

<a id="test-7"></a>
## Test 7: KV Cache Cliff — Long Context (Stage 2)

### Scenario
Push KV cache toward capacity with long-context requests. Each request: ~500 input tokens + 1500 output tokens = ~2000 total. Same prompt for all requests (prefix caching ON). Ramp N = 20 → 70.

Theoretical max at 2000 tokens/request: 127,840 / 2000 ≈ 64. But prefix caching means the shared 500-token prefix is stored once, reducing effective per-request KV to ~1500 tokens.

In [ ]:
# Test 7: KV Cache cliff test — throughput + KV usage
valid_cliff = [r for r in cliff if r.get('successes', 0) > 0]
ns_c = [r['concurrency'] for r in valid_cliff]
sys_c = [r['system_tok_s'] for r in valid_cliff]
req_c = [r['per_request_tok_s_avg'] for r in valid_cliff]
total_c = [r['total_p50'] for r in valid_cliff]

# Extract KV peak from server logs
kv_peaks = []
for r in valid_cliff:
    logs = r.get('server_logs', '')
    # Parse KV cache usage from log strings
    import re
    matches = re.findall(r'KV cache usage: ([\d.]+)%', str(logs))
    kv_peaks.append(max([float(m) for m in matches]) if matches else 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: System throughput + per-request
ax1.plot(ns_c, sys_c, 'o-', color='#3498db', linewidth=2, markersize=8, label='System tok/s')
ax1.set_xlabel('Concurrent Requests')
ax1.set_ylabel('System Throughput (tok/s)', color='#3498db')

ax1b = ax1.twinx()
ax1b.plot(ns_c, total_c, 's--', color='#e74c3c', linewidth=2, markersize=8, label='Total time p50 (s)')
ax1b.set_ylabel('Total Request Time (seconds)', color='#e74c3c')

peak_idx = sys_c.index(max(sys_c))
ax1.annotate(f'Peak: {max(sys_c):.0f} tok/s\nat N={ns_c[peak_idx]}', 
            xy=(ns_c[peak_idx], max(sys_c)), xytext=(ns_c[peak_idx]-10, max(sys_c)+30),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
ax1.set_title('Throughput Peaks, Then Request Time Keeps Growing')

# Chart 2: KV cache usage
if any(k > 0 for k in kv_peaks):
    ax2.bar(ns_c, kv_peaks, color=['#2ecc71' if k < 80 else '#f39c12' if k < 95 else '#e74c3c' for k in kv_peaks], alpha=0.8)
    ax2.axhline(y=100, color='red', linestyle='--', alpha=0.7, label='KV Cache Full')
    ax2.set_xlabel('Concurrent Requests')
    ax2.set_ylabel('Peak KV Cache Usage (%)')
    ax2.set_title('KV Cache Usage — Approaching the Wall')
    ax2.legend()
else:
    # Fallback if KV data not captured in logs
    ax2.text(0.5, 0.5, 'KV peak data from server logs:\n\n' + 
             '\n'.join([f'N={n}: {r.get("server_logs", "no logs")[:60]}...' for n, r in zip(ns_c[-3:], valid_cliff[-3:])]),
             transform=ax2.transAxes, ha='center', va='center', fontsize=9, family='monospace')
    ax2.set_title('KV Cache Progression (from server logs)')

plt.suptitle('Test 7: Long-Context Requests — Pushing KV Cache Toward Capacity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Peak system throughput: {max(sys_c):.0f} tok/s at N={ns_c[sys_c.index(max(sys_c))]}")
print(f"Prefix cache absorbed most prefill pressure (93% hit rate by N=70)")

### Insights
- Peak at N=60 (900 tok/s) with long-context — same optimal point as Test 5
- KV cache reached 78.4% at N=70 but **no preemption, no queuing** — prefix cache (93% hit rate) reduced effective per-request KV from ~2000 to ~1500 tokens
- Total request time grew from 56s (N=20) to 132s (N=70) — linear with concurrency
- Zero failures across all rounds — vLLM handled everything within KV budget

**Why we didn't hit the wall:** Prefix caching stores the shared ~500-token prompt once. 70 requests × 1500 unique tokens = 105K tokens, well within 127,840 capacity.

### Your Notes
*Add your own observations here*

---

<a id="test-8"></a>
## Test 8: Preemption & Queuing — Breaking the Server (Stage 2)

### Scenario
Hardest conditions: **GPTQ Int8** (8.3GB weights → only 4.21GB KV cache → max 38 concurrent), **prefix caching OFF**, **40 unique prompts** (no KV sharing possible), max_tokens=1500. We fired 40 new requests while ~35 were still running, creating ~75 total against a max of ~39.

This is the test that finally broke vLLM.

In [ ]:
# Test 8: Preemption & queuing — KV cache timeline from server logs
# This data was captured manually from docker logs during the overload burst

kv_timeline = [
    ('03:58:42', 77, 0, 91.8),
    ('03:58:52', 77, 0, 96.5),
    ('03:59:02', 76, 0, 99.7),
    ('03:59:12', 73, 1, 99.8),   # First queued request!
    ('03:59:22', 70, 3, 100.0),  # KV FULL
    ('03:59:32', 68, 2, 99.0),
    ('03:59:42', 66, 2, 99.4),
    ('03:59:52', 63, 4, 99.0),
    ('04:00:02', 61, 5, 99.5),   # Peak waiting
    ('04:00:12', 38, 0, 52.9),   # Burst drained
]

times = list(range(len(kv_timeline)))
labels = [t[0].split(':')[1] + ':' + t[0].split(':')[2] for t in kv_timeline]
running = [t[1] for t in kv_timeline]
waiting = [t[2] for t in kv_timeline]
kv_pct = [t[3] for t in kv_timeline]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Chart 1: Running + Waiting requests
ax1.fill_between(times, running, alpha=0.4, color='#3498db', label='Running')
ax1.fill_between(times, [r + w for r, w in zip(running, waiting)], running, 
                alpha=0.7, color='#e74c3c', label='Waiting (QUEUED)')
ax1.plot(times, running, 'o-', color='#3498db', linewidth=2)
ax1.plot(times, [r + w for r, w in zip(running, waiting)], 's-', color='#e74c3c', linewidth=2)
ax1.set_ylabel('Request Count')
ax1.set_title('Running vs Waiting Requests')
ax1.legend()
ax1.axhline(y=39, color='gray', linestyle='--', alpha=0.5)
ax1.text(0.5, 41, 'Theoretical max (39)', fontsize=9, color='gray')

# Annotate first queue event
ax1.annotate('First queued\nrequest!', xy=(3, 74), xytext=(1, 80),
            arrowprops=dict(arrowstyle='->', color='red', lw=2), 
            fontsize=11, color='red', fontweight='bold')

# Chart 2: KV cache usage
colors_kv = ['#2ecc71' if k < 95 else '#f39c12' if k < 99.5 else '#e74c3c' for k in kv_pct]
ax2.bar(times, kv_pct, color=colors_kv, alpha=0.8, edgecolor='white')
ax2.axhline(y=100, color='red', linestyle='--', linewidth=2, alpha=0.7, label='100% — No room for new requests')
ax2.set_ylabel('KV Cache Usage (%)')
ax2.set_xlabel('Time')
ax2.set_xticks(times)
ax2.set_xticklabels(labels, rotation=45)
ax2.set_title('KV Cache Usage — Hitting 100%')
ax2.set_ylim(0, 110)
ax2.legend()

# Annotate the 100% moment
full_idx = next(i for i, k in enumerate(kv_pct) if k >= 100)
ax2.annotate('KV CACHE FULL', xy=(full_idx, 100), xytext=(full_idx+2, 105),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=11, color='red', fontweight='bold')

plt.suptitle('Test 8: KV Cache at 100% — vLLM Queues New Requests', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Also show the sequential rounds from the subagent
print("\nSequential rounds (from subagent):")
for r in preemption:
    if r.get('ok', 0) > 0:
        print(f"  N={r['n']:>2} | OK: {r['ok']:>2} | TTFT p50: {r['ttft_p50']:.1f}s | "
              f"Total p50: {r['total_p50']:.0f}s | tok/s: {r['avg_tok_per_sec']:.1f}")
    else:
        print(f"  N={r['n']:>2} | FAILED — {r.get('errors', ['server crashed'])[0][:50] if r.get('errors') else 'server crashed'}")

### Insights
- **KV hit 100%** — the first time in all our tests. Triggered by: Int8 (less KV room) + no prefix cache + unique prompts
- **vLLM chose queuing over preemption:** `Waiting` went from 0 → 5. Running requests kept their KV pages. New requests waited for pages to free up as running requests completed.
- **No preemption logged** — evicting a running request's KV (throwing away minutes of decode work) is more wasteful than delaying a new request by seconds
- **Sequential runs confirmed theoretical max:** N=38 succeeded, N=40 barely succeeded, N=45 crashed the server
- **Failure mode past the limit is catastrophic, not graceful** — the server crashed entirely rather than returning 503s

**Auto Scaling analogy:** ASG at max capacity. New requests queue behind the ALB. You don't kill healthy instances to make room — you wait for natural draining. `Waiting > 0` is your scaling signal for Stage 4.

### Your Notes
*Add your own observations here*

---

<a id="test-9"></a>
## Test 9: Prefill vs Decode — Different Bottlenecks (Stage 2)

### Scenario
Two sub-tests on AWQ 4-bit, single request (no concurrency), each point averaged over 3 trials:
- **Test A:** Fixed output (50 tokens), vary input from 10 → 1000 tokens
- **Test B:** Fixed input (~50 tokens), vary output from 10 → 1000 tokens

We want to prove prefill time depends on input size (compute-bound) while decode time depends on output size (memory-bandwidth-bound), and they're independent of each other.

In [ ]:
# Test 9: Prefill vs Decode — independent bottlenecks
test_a = exp2['test_a_vary_input']
test_b = exp2['test_b_vary_output']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Test A — vary input, fixed output
input_toks = [r['input_tokens_target'] for r in test_a]
ttfts_a = [r['avg_ttft_sec'] for r in test_a]
decode_a = [r['avg_decode_time_sec'] for r in test_a]

ax1.plot(input_toks, ttfts_a, 'o-', color='#3498db', linewidth=2, markersize=8, label='TTFT (prefill)')
ax1.plot(input_toks, decode_a, 's-', color='#e74c3c', linewidth=2, markersize=8, label='Decode time')
ax1.set_xlabel('Input Tokens')
ax1.set_ylabel('Time (seconds)')
ax1.set_title('Test A: Vary Input (fixed 50 output tokens)\nTTFT barely changes — decode is flat')
ax1.legend()
ax1.set_xscale('log')

# Annotate the key finding
ax1.annotate(f'100x more input\n→ {ttfts_a[-1]/ttfts_a[0]:.1f}x TTFT change', 
            xy=(500, max(ttfts_a)), xytext=(100, max(decode_a)*0.8),
            fontsize=10, color='#3498db', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#3498db'))

# Chart 2: Test B — fixed input, vary output
output_toks = [r['avg_tokens_out'] for r in test_b]
ttfts_b = [r['avg_ttft_sec'] for r in test_b]
decode_b = [r['avg_decode_time_sec'] for r in test_b]

ax2.plot(output_toks, ttfts_b, 'o-', color='#3498db', linewidth=2, markersize=8, label='TTFT (prefill)')
ax2.plot(output_toks, decode_b, 's-', color='#e74c3c', linewidth=2, markersize=8, label='Decode time')
ax2.set_xlabel('Output Tokens (actual)')
ax2.set_ylabel('Time (seconds)')
ax2.set_title('Test B: Vary Output (fixed ~50 input tokens)\nDecode scales linearly — TTFT is flat')
ax2.legend()

# Annotate
if len(decode_b) >= 2 and decode_b[0] > 0:
    ax2.annotate(f'{output_toks[-1]/output_toks[0]:.0f}x more output\n→ {decode_b[-1]/decode_b[0]:.1f}x decode time', 
                xy=(output_toks[-1], decode_b[-1]), xytext=(output_toks[-1]*0.3, decode_b[-1]*0.7),
                fontsize=10, color='#e74c3c', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#e74c3c'))

plt.suptitle('Test 9: Prefill and Decode Are Independent Bottlenecks', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Insights
- **Test A:** 100x input increase (10→1000 tokens) → TTFT changed by only 1.0x. Decode time unchanged (~1.1s). Prefill absorbs input growth through parallel GPU computation.
- **Test B:** ~17x output increase → 18.7x decode time increase. TTFT stayed flat at ~0.21s. Each output token costs the same fixed time (~22ms).
- These are **independent bottlenecks**: doubling input doesn't slow decode, doubling output doesn't slow prefill
- Model stopped early on high output targets (got 168 instead of 1000) — prompt wasn't compelling enough for long generation

**Auto Scaling analogy:** Prefill = MapReduce batch job (adding data barely increases wall time due to parallelism). Decode = sequential pipeline (each stage takes fixed time, total = N × stage_time).

### Your Notes
*Add your own observations here*

---

<a id="test-10"></a>
## Test 10: Why Output Tokens Cost More Than Input Tokens (Stage 2)

### Scenario
Three sub-tests measuring the actual GPU cost asymmetry between input and output tokens:
1. Same ~1000 total tokens, different input/output splits (900in+100out vs 100in+900out)
2. GPU-milliseconds per token for input vs output at various sizes
3. Raw throughput: prefill tok/s vs decode tok/s

This explains why API providers charge 3-5x more for output tokens.

In [ ]:
# Test 10: Why output tokens cost more
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Chart 1: Same total tokens, different split
test1 = exp3['test1_split_comparison']
labels_t1 = [r['label'] for r in test1]
prefill_t1 = [r['prefill_sec'] for r in test1]
decode_t1 = [r['decode_sec'] for r in test1]
total_t1 = [r['total_sec'] for r in test1]

x = np.arange(len(labels_t1))
axes[0].bar(x, prefill_t1, 0.6, label='Prefill', color='#3498db', alpha=0.8)
axes[0].bar(x, decode_t1, 0.6, bottom=prefill_t1, label='Decode', color='#e74c3c', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{r['label']}\n({r['input']}in+{r['output']}out)" for r in test1], fontsize=9)
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Same ~1000 Total Tokens\nDifferent Split → Vastly Different Time')
axes[0].legend()

for i, t in enumerate(total_t1):
    axes[0].text(i, t + 0.3, f'{t:.1f}s', ha='center', fontweight='bold')

# Chart 2: Cost per token — input vs output
test2a = exp3['test2a_cost_per_input_token']
test2b = exp3['test2b_cost_per_output_token']

input_sizes = [r['input_tokens'] for r in test2a]
input_costs = [r['cost_per_input_token_ms'] for r in test2a]
output_sizes = [r.get('output_tokens', r.get('tokens_out', 0)) for r in test2b]
output_costs = [r['cost_per_output_token_ms'] for r in test2b]

axes[1].plot(input_sizes, input_costs, 'o-', color='#3498db', linewidth=2, markersize=8, label='Input (prefill)')
axes[1].axhline(y=np.mean(output_costs), color='#e74c3c', linestyle='--', linewidth=2, alpha=0.7, 
               label=f'Output avg ({np.mean(output_costs):.1f} ms/tok)')
axes[1].set_xlabel('Token Count')
axes[1].set_ylabel('GPU Cost (ms/token)')
axes[1].set_title('Cost Per Token\nInput gets CHEAPER at scale')
axes[1].legend()
axes[1].set_xscale('log')

# Annotate the ratio
avg_input = np.mean(input_costs)
avg_output = np.mean(output_costs)
axes[1].text(200, avg_output * 0.7, f'Output costs\n{avg_output/avg_input:.0f}x more\nthan input', 
            fontsize=11, color='red', fontweight='bold', ha='center')

# Chart 3: Throughput comparison
test3 = exp3['test3_throughput']
throughputs = [test3['prefill_throughput_tok_s'], test3['decode_throughput_tok_s']]
labels_t3 = ['Prefill\n(parallel)', 'Decode\n(sequential)']
colors_t3 = ['#3498db', '#e74c3c']

bars = axes[2].bar(labels_t3, throughputs, color=colors_t3, alpha=0.8, width=0.5)
axes[2].set_ylabel('Tokens/second')
axes[2].set_title(f'Raw Throughput\nPrefill is {throughputs[0]/throughputs[1]:.0f}x Faster')
axes[2].set_yscale('log')

for bar, val in zip(bars, throughputs):
    axes[2].text(bar.get_x() + bar.get_width()/2, val * 1.5, f'{val:.0f}', 
                ha='center', fontweight='bold', fontsize=12)

plt.suptitle('Test 10: Why API Providers Charge More for Output Tokens', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nSummary:")
print(f"  Average GPU cost per INPUT token:  {avg_input:.2f} ms")
print(f"  Average GPU cost per OUTPUT token: {avg_output:.2f} ms")
print(f"  Output tokens cost {avg_output/avg_input:.1f}x more GPU time")
print(f"  Prefill throughput: {throughputs[0]:,.0f} tok/s vs Decode: {throughputs[1]:.0f} tok/s ({throughputs[0]/throughputs[1]:.0f}x ratio)")

### Insights
- **Same 1000 tokens, 6x time difference:** Heavy input (900+100) = 2.6s. Heavy output (100+900) = 15.6s. Input and output tokens are NOT equivalent.
- **Output tokens cost 16.9x more GPU time** than input tokens (21.6 ms vs 1.3 ms per token)
- **Input cost DECREASES at scale** — 50 tokens = 3.6 ms/tok, 1000 tokens = 0.2 ms/tok. Parallel processing amortizes overhead.
- **Output cost is CONSTANT** — 21.1 ms/tok whether you generate 50 or 954 tokens. Each token reads the full model weights.
- **Prefill is 109x faster per token** than decode (4,983 vs 46 tok/s)
- This is why Anthropic charges ~5x more for output tokens — the GPU cost difference is real, not arbitrary. Our 16.9x measurement is on a T4; production H100s have higher memory bandwidth, reducing the ratio but not eliminating it.

**Root cause:** Input tokens = parallel matrix-matrix multiply (GPU compute units fully utilized). Output tokens = sequential matrix-vector multiply (GPU mostly idle, waiting for memory reads). The GPU sits idle ~95% of the time during decode.

### Your Notes
*Add your own observations here*

---

## Hardware & Environment Reference

| Component | Stage 1 | Stage 2 |
|-----------|---------|---------|
| Hardware | Apple M4, 16GB unified | AWS g4dn.xlarge, NVIDIA T4 16GB |
| Memory BW | ~100 GB/s | 320 GB/s |
| Engine | Ollama (llama.cpp) | vLLM v0.17.1 |
| Model | Qwen2.5:7B Q4_K_M | Qwen2.5-7B-Instruct-AWQ / GPTQ-Int8 |
| Spot Cost | N/A (local) | ~$0.22/hr |